In [ ]:
"""
Feature importance analysis for the v4 model.

Uses permutation importance: for each feature, we shuffle its values
across pixels and time steps, then measure how much the AUC drops.
Features that cause a large AUC drop when shuffled are the most
important for burned month detection.

Run this as a standalone notebook cell after Notebook 05 has completed.
"""

# Feature Importance Analysis (v4)

Measures which of the 52 features contribute most to burned month
detection using permutation importance.

In [ ]:
# --- Environment auto-detection ---
import sys, os
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    THESIS_ROOT = "/content/drive/MyDrive/thesisv4"
else:
    THESIS_ROOT = os.path.expanduser("~/thesisv4")

sys.path.insert(0, THESIS_ROOT)
print(f"=== Environment: {'COLAB' if IS_COLAB else 'LOCAL'} ===")
print(f"Workspace: {THESIS_ROOT}")

In [ ]:
import sys, importlib
# sys.path already set by auto-detect cell above
import config, model_defs, metrics
for m in (config, model_defs, metrics):
    importlib.reload(m)

In [ ]:
# --- Check which required files are present locally ---
import os

REQUIRED_FILES = {
    "data/processed/X_scaled.npy":         13_000,
    "data/processed/X_scaled_mask.npy":     2_000,
    "data/processed/y_burned.npy":             65,
    "data/processed/targets_aux.npz":          10,
    "data/processed/splits.json":               1,
    "data/processed/feature_names.json":        1,
    "data/processed/scaler.pkl":                1,
    "models/main/model_main.pt":                1,
    "models/main/results_main.json":            1,
}

def check_files():
    missing = []
    for rel_path in REQUIRED_FILES:
        full_path = os.path.join(THESIS_ROOT, rel_path)
        if not os.path.exists(full_path):
            missing.append(rel_path)
    return missing

if IS_COLAB:
    print("Running on Colab — files already on mounted Drive. Skipping check.")
    missing = []
else:
    missing = check_files()
    if not missing:
        print(f"All {len(REQUIRED_FILES)} required files present locally.")
    else:
        print(f"Missing {len(missing)} of {len(REQUIRED_FILES)} required files:")
        total_mb = sum(REQUIRED_FILES[f] for f in missing)
        for f in missing:
            print(f"  - {f}   (~{REQUIRED_FILES[f]} MB)")
        print(f"\nTotal to download: ~{total_mb/1024:.1f} GB")


In [ ]:
# --- Auto-download from Drive via rclone (local only, skipped on Colab) ---
# Prerequisites (one-time setup):
#   1. Install rclone:
#      - Ubuntu/Debian: sudo apt install rclone
#      - macOS:          brew install rclone
#      - Windows:        download from https://rclone.org/downloads/
#   2. Configure a Google Drive remote named 'gdrive':
#         rclone config    (choose 'n', name 'gdrive', type 'drive', scope 1)
#   3. Verify with:
#         rclone lsd gdrive:

import subprocess

RCLONE_REMOTE = "gdrive"
DRIVE_ROOT    = "thesisv4"

if IS_COLAB:
    print("Running on Colab — files already on mounted Drive. Skipping download.")
elif not missing:
    print("All required files already present locally. Skipping download.")
else:
    try:
        subprocess.run(["rclone", "version"],
                       check=True, capture_output=True, text=True)
    except (subprocess.CalledProcessError, FileNotFoundError):
        raise SystemExit(
            "rclone is not installed or not in PATH. "
            "Install: https://rclone.org/downloads/")

    check = subprocess.run(
        ["rclone", "listremotes"], capture_output=True, text=True)
    if f"{RCLONE_REMOTE}:" not in check.stdout:
        raise SystemExit(
            f"rclone remote '{RCLONE_REMOTE}' not configured. "
            f"Run 'rclone config' first.")

    print(f"Downloading {len(missing)} missing files via rclone...")
    print(f"Total: ~{sum(REQUIRED_FILES[f] for f in missing) / 1024:.1f} GB")
    print("Expected time: 30-90 minutes.\n")

    for rel_path in missing:
        remote_path = f"{RCLONE_REMOTE}:{DRIVE_ROOT}/{rel_path}"
        local_path  = os.path.join(THESIS_ROOT, rel_path)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)

        print(f"Downloading {rel_path}  (~{REQUIRED_FILES[rel_path]} MB)")
        result = subprocess.run(
            ["rclone", "copyto", remote_path, local_path,
             "--progress", "--transfers", "4", "--checkers", "4"],
            check=False,
        )
        if result.returncode != 0:
            raise RuntimeError(f"rclone failed for {rel_path}")
        size_mb = os.path.getsize(local_path) / 1024**2
        print(f"  Done — {size_mb:.1f} MB\n")

    print(f"All {len(missing)} files downloaded to {THESIS_ROOT}")

print("\nReady to load data.")


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
# Load everything the trained model needs
X       = np.load(config.FILES["X_scaled"])
y       = np.load(config.FILES["y_burned"])
mask    = np.load(config.FILES["X_scaled"].replace(".npy", "_mask.npy"))
targets = dict(np.load(config.FILES["targets_aux"]))

with open(config.FILES["splits"]) as f:
    splits = json.load(f)
with open(config.FILES["feature_names"]) as f:
    feature_names = json.load(f)

idx_test = np.array(splits["test"])

N, T, C = X.shape
print(f"X shape:       {X.shape}")
print(f"Test pixels:   {len(idx_test)}")
print(f"Feature count: {C}")

In [ ]:
# Rebuild and load the trained model
model = model_defs.FireRecoveryModel(
    n_features=C,
    d_model=config.MODEL["d_model"],
    n_head=config.MODEL["n_head"],
    d_k=config.MODEL["d_k"],
    max_len=T,
    tau=config.MODEL["positional_tau"],
    dropout=0.0,
).to(DEVICE)

state = torch.load(config.FILES["model_main"], map_location=DEVICE)
model.load_state_dict(state)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {n_params:,} parameters")

In [ ]:
def predict_auc(X_input, y_input, mask_input, indices, batch_size=128):
    """Compute AUC on the specified pixel indices."""
    model.eval()
    probs_all  = []
    labels_all = []
    mask_all   = []

    with torch.no_grad():
        for i in range(0, len(indices), batch_size):
            batch_idx = indices[i:i + batch_size]
            Xb = torch.tensor(X_input[batch_idx]).to(DEVICE)
            Mb = torch.tensor(mask_input[batch_idx]).to(DEVICE)
            out = model(Xb, mask=Mb)
            probs_all.append(
                torch.sigmoid(out["burned_logits"]).cpu().numpy())
            labels_all.append(y_input[batch_idx])
            mask_all.append(mask_input[batch_idx])

    probs  = np.concatenate(probs_all).reshape(-1)
    labels = np.concatenate(labels_all).reshape(-1)
    msk    = np.concatenate(mask_all).reshape(-1).astype(bool)
    return roc_auc_score(labels[msk], probs[msk])

## Compute baseline AUC on test set

In [ ]:
baseline_auc = predict_auc(X, y, mask, idx_test)
print(f"Baseline AUC on test set: {baseline_auc:.4f}")

## Compute permutation importance for each feature

For each of the 52 features, we shuffle its values across all pixels
and time steps, then measure how much AUC drops. Runs in about 5-15
minutes on GPU.

In [ ]:
NUM_REPEATS = 3   # more repeats give more stable estimates
rng = np.random.RandomState(config.SEED)

importance_scores = np.zeros(C, dtype=np.float64)

for feature_idx in tqdm(range(C), desc="Features"):
    aucs_after = []
    for _ in range(NUM_REPEATS):
        # Shuffle only this feature in place, then restore it even if inference
        # fails. This preserves the permutation distribution without copying
        # the entire 13+ GB input tensor for every repeat.
        original_values = X[:, :, feature_idx].copy()
        shuffled_values = original_values.copy()
        rng.shuffle(shuffled_values.reshape(-1))
        try:
            X[:, :, feature_idx] = shuffled_values
            aucs_after.append(predict_auc(X, y, mask, idx_test))
        finally:
            X[:, :, feature_idx] = original_values

    mean_auc_after = np.mean(aucs_after)
    importance_scores[feature_idx] = baseline_auc - mean_auc_after


## Rank features by importance

In [ ]:
importance_df = pd.DataFrame({
    "feature":    feature_names,
    "importance": importance_scores,
}).sort_values("importance", ascending=False)

# Family label — spectral, delta, harmonic, or LC
def feature_family(name):
    if name in config.LC_FEATURES:       return "land_cover"
    if name in config.DELTA_FEATURES:    return "delta"
    if name in config.HARMONIC_FEATURES: return "harmonic"
    if "_zscore" in name or "_anomaly" in name: return "anomaly"
    if name in config.INDICES:           return "index"
    if name in config.RAW_BANDS:         return "raw_band"
    return "other"

importance_df["family"] = importance_df["feature"].apply(feature_family)

print("Top 20 most important features:")
print(importance_df.head(20).to_string(index=False))

In [ ]:
print("\nBottom 10 (least important) features:")
print(importance_df.tail(10).to_string(index=False))

## Aggregate by family

In [ ]:
family_summary = (importance_df
                  .groupby("family")
                  .agg(mean_importance=("importance", "mean"),
                       max_importance=("importance", "max"),
                       n_features=("feature", "count"))
                  .sort_values("mean_importance", ascending=False))
print("\nImportance aggregated by feature family:")
print(family_summary.to_string())

## Visualise the top features

In [ ]:
top_n = 20
top_features = importance_df.head(top_n)

fig, ax = plt.subplots(figsize=(9, 8))

family_colors = {
    "delta":      "#e74c3c",
    "anomaly":    "#3498db",
    "index":      "#2ecc71",
    "raw_band":   "#f39c12",
    "harmonic":   "#9b59b6",
    "land_cover": "#95a5a6",
}
colors = [family_colors[f] for f in top_features["family"]]

ax.barh(top_features["feature"], top_features["importance"], color=colors)
ax.set_xlabel("AUC drop when shuffled (higher = more important)")
ax.set_title(f"Top {top_n} feature importance (baseline AUC = {baseline_auc:.3f})")
ax.invert_yaxis()

from matplotlib.patches import Patch
legend_handles = [Patch(color=c, label=fam)
                  for fam, c in family_colors.items()]
ax.legend(handles=legend_handles, loc="lower right")

plt.tight_layout()
plt.savefig(f"{config.PATHS['outputs_fig']}/feature_importance_top{top_n}.png",
            dpi=150, bbox_inches="tight")
plt.show()

## Save results for the thesis

In [ ]:
importance_path = f"{config.PATHS['outputs_tab']}/feature_importance.csv"
importance_df.to_csv(importance_path, index=False)
print(f"Saved feature importance to: {importance_path}")

family_path = f"{config.PATHS['outputs_tab']}/feature_importance_by_family.csv"
family_summary.to_csv(family_path)
print(f"Saved family aggregation to: {family_path}")